In [1]:
#importing all module which is used in this project.
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import nltk
nltk.download('punkt_tab')
from torch.utils.data import DataLoader,Dataset

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
#now laoding the sequential dataset 
data = pd.read_csv('100_Unique_QA_Dataset.csv')
data.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


# applying text preprocessing to sequential dataset converting them into word tokens

In [3]:
#calling tokenizer class of nltk
from nltk.tokenize import word_tokenize
import re

def text_to_tokens(sentence):
    #before changing the sentence to word tokens first of all we have to do cleaning.
    email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
    str_special = r'[^a-zA-Z0-9\s-]'
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    html_pattern = r'<[^>]*>.*?</[^>]*>'
    
    #changing the senetence to lower case.
    sentence = sentence.lower()
    
    #applying regular expression pattern to sentence.
    sentence = re.sub(email_pattern,"",sentence)
    sentence = re.sub(str_special,"",sentence)
    sentence = re.sub(url_pattern,"",sentence)
    sentence = re.sub(html_pattern,"",sentence)
    
    
    #Once cleaning done now applying word tokenizer to senetence.
    lst_words = word_tokenize(sentence)
    return lst_words
    

In [4]:
#calling the text to token function checking fucntion is working correct or not.
data.question.apply(text_to_tokens)[:5]

0                 [what, is, the, capital, of, france]
1                [what, is, the, capital, of, germany]
2               [who, wrote, to, kill, a, mockingbird]
3    [what, is, the, largest, planet, in, our, sola...
4    [what, is, the, boiling, point, of, water, in,...
Name: question, dtype: object

# before applying integer encoding to sequential data first of all we have to build the vocubolary

In [5]:
#vocabolary means collection of unique words.
vocab = {"<UNK>":0}

#to build the vocabolary object we have to pass entire record from dataset that include input and output variables.

def build_vocabolary(row):
    #changing input and output tokens to words
    question = text_to_tokens(row.question)
    answer   = text_to_tokens(row.answer)
    
    #Now concatenation both lst object
    merger_row = question + answer
    for word in merger_row:
        if word not in vocab:
            vocab[word] = len(vocab)   #then assign word ko position number.
            

#calling function and checking vocab is creating or not.
_ = data.apply(build_vocabolary,axis=1)

In [6]:
#if i want to show based on the data we have have how many unique vocabolary was created.
len(vocab) 
#above vocabolary ke andar we have word and its index position number using them we gonna create integer encoding to data.

324

# now applying integer encoding to sequential data based on vocabolary we have on both variable 

In [7]:
def integer_encoding(row,vocab):
    text_encoding = []
    for word in text_to_tokens(row):
        if word in vocab:
            text_encoding.append(vocab[word])
        else:
            text_encoding.append(vocab['<UNK>'])
    
    return text_encoding

# Now creating Custom datset class

In [8]:
class QADataset(Dataset):
    #constructor method we used to initalize the instance variable.
    def __init__(self,data,vocab):
        super().__init__()
        self.data = data
        self.vocab = vocab
        
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        #this special method (__getitem__) based on index selecting input and output variable.
        question = integer_encoding(data.iloc[index]['question'],self.vocab)
        answer   = integer_encoding(data.iloc[index]['answer'],self.vocab)
        
        #changing them into tensor object
        return torch.tensor(question,dtype=torch.long),torch.tensor(answer,dtype=torch.long)
  

In [9]:
#creating an object of custom dataset class.
seq_data = QADataset(data,vocab)
seq_data

# using dataloader class we are loading data into batch_size.

In [10]:
#creating an object of dataloader class
seq_dataloader = DataLoader(seq_data,batch_size=1,shuffle=True,pin_memory=True)

In [11]:
#checking how the data will look like after done integer encoding!!!
for question,answer in seq_dataloader:
    print(question)
    print(answer)
    break

tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]])
tensor([[321]])


In [12]:
'''
Default behavior (batch_first=False)
Input ka expected shape hota hai:
(sequence_length, batch_size, input_size) #input_size means vocobolary size 
Matlab pehla dimension time steps hota hai, fir batch, fir feature size.

But
When batch_first=True
Input ka expected shape hota hai:
(batch_size, sequence_length, input_size) #input_size means vocobolary size 
Matlab pehla dimension batch hota hai, fir sequence length, fir feature size.

'''

'\nDefault behavior (batch_first=False)\nInput ka expected shape hota hai:\n(sequence_length, batch_size, input_size) #input_size means vocobolary size \nMatlab pehla dimension time steps hota hai, fir batch, fir feature size.\n\nBut\nWhen batch_first=True\nInput ka expected shape hota hai:\n(batch_size, sequence_length, input_size) #input_size means vocobolary size \nMatlab pehla dimension batch hota hai, fir sequence length, fir feature size.\n\n'

# defining neural network architecture

In [13]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [14]:
#creating an object of Custom RNN Neural Network Architecture.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = len(vocab)
model = SimpleRNN(vocab_size=vocab_size).to(device)
model

SimpleRNN(
  (embedding): Embedding(324, 50)
  (rnn): RNN(50, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=324, bias=True)
)

In [15]:
#if i want to see how many trainable parameter will be train by model
from torchinfo import summary
summary(model=model)

Layer (type:depth-idx)                   Param #
SimpleRNN                                --
├─Embedding: 1-1                         16,200
├─RNN: 1-2                               7,424
├─Linear: 1-3                            21,060
Total params: 44,684
Trainable params: 44,684
Non-trainable params: 0

# now defining learning rate epochs and optimizer 

In [16]:
learning_rate = 0.01
optimizers = torch.optim.Adam(model.parameters(),lr=learning_rate,weight_decay=0.001)
#optimizer will update weights and bias in each layer at time of training
epochs = 50

#now defining the loss function for multiclassifer and its inter encode so we are using sparse_categorical cross entropy
loss_fxn = torch.nn.CrossEntropyLoss()
loss_fxn

CrossEntropyLoss()

# now starting the model training

In [17]:
for epoch in range(epochs):
    model.train()
    training_loss_count = 0
    
    #first we loading the dataset then passing them into neural network architecture.
    for question,answer in seq_dataloader:
        question = question.to(device)
        answer   = answer.to(device)
        
        #passing input sequential to architecture first algo that work is forward pass geenrate the predicted output.
        prediction = model.forward(question) #prediction of output expected in this shape [batch_size,voc_size]
        #but we are getting [batch-size,seq_lenth,voc_size] so thatswhy we have done changes in last_layer.
  
        answer = answer.squeeze(0) #label shape changing from 2d--->1d tensor object.
        
        optimizers.zero_grad()
        
        #calculating the loss value
        losses = loss_fxn(prediction,answer)
        
        losses.backward()
        
        optimizers.step()
        
        training_loss_count = training_loss_count+losses
    print(f"epochs: {epoch+1} and train_loss: {training_loss_count}")
  

epochs: 1 and train_loss: 547.052490234375
epochs: 2 and train_loss: 481.9052734375
epochs: 3 and train_loss: 392.0132141113281
epochs: 4 and train_loss: 292.31500244140625
epochs: 5 and train_loss: 218.5065460205078
epochs: 6 and train_loss: 130.49444580078125
epochs: 7 and train_loss: 102.63009643554688
epochs: 8 and train_loss: 51.952327728271484
epochs: 9 and train_loss: 33.696083068847656
epochs: 10 and train_loss: 21.920347213745117
epochs: 11 and train_loss: 25.42548179626465
epochs: 12 and train_loss: 18.858871459960938
epochs: 13 and train_loss: 20.711668014526367
epochs: 14 and train_loss: 20.294946670532227
epochs: 15 and train_loss: 25.324506759643555
epochs: 16 and train_loss: 115.72217559814453
epochs: 17 and train_loss: 146.56033325195312
epochs: 18 and train_loss: 134.35220336914062
epochs: 19 and train_loss: 125.19180297851562
epochs: 20 and train_loss: 63.512489318847656
epochs: 21 and train_loss: 43.149169921875
epochs: 22 and train_loss: 22.829059600830078
epochs: 2

# now testing the model with test question

In [25]:
#agar mere model ko lagegha ki usne question dekh hai at time of training to prediction degha varna i dont know answer degha.
def predict(model,question,threshold=0.5):
    #converting question to integer encoding
    numerical_question = integer_encoding(question,vocab)
    
    #changing them into tensor object
    question_tensor = torch.tensor(numerical_question, dtype=torch.long).unsqueeze(0).to(device)  #shape we wont [batch_size,seq_length]
    
    #sending the input seq data to model
    output = model(question_tensor)
    
    #converting logits to probability
    prob = torch.nn.functional.softmax(output,dim=1)
    
    #finding max probability
    value,index = torch.max(prob,1)
    
    if value < threshold:
        print("I don't know")

    else:
        print(list(vocab.keys())[index])


In [27]:
predict(model,'What is the longest river in the world?')

nile
